In [ ]:
# 2024-03-24
import pandas as pd
from tqdm import tqdm
from normdata.utils import import_from_normdata
from csae_pyutils import gsheet_to_df
from apis_core.apis_metainfo.models import Collection, CollectionType, Uri
from apis_core.apis_entities.models import Person

In [ ]:
df = gsheet_to_df("1BRRF46M_XP1HTRIY4mF_KFn9RB3ECLCDIebmB7lxwj4")

In [ ]:
df

In [ ]:
col, _ = Collection.objects.get_or_create(name="S-Fischer")
domain = "sfischer"
col_type, _ = CollectionType.objects.get_or_create(name="Projekt")
col.description = 'To bed onde'
col.collection_type = col_type
col.save()

In [ ]:
print("process orgs with gnd")
broken_gnd = []
for i, row in tqdm(df.iterrows()):
    tmp_uri = f"http://sfischer/foo/bar/org/{i+1}"
    if pd.isna(row["gnd_id"]):
        continue
    else:
        gnd = row["gnd_id"]
        gnd_uri = f"https://d-nb.info/gnd/{gnd}"
        entity = import_from_normdata(gnd_uri, "institution")
        if entity:
            pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
            pmb_uri.entity = entity
            pmb_uri.save()
            entity.collection.add(col)
        else:
            broken_gnd.append([tmp_uri, gnd_uri])

In [ ]:
broken_gnd